In [1]:
def character_type_analyzer(text):
    vowels = "aeiouAEIOU"
    
    result = {
        "vowels": 0,
        "consonants": 0,
        "digits": 0,
        "spaces": 0
    }
    
    for char in text:
        if char in vowels:
            result["vowels"] += 1
        elif char.isalpha():
            result["consonants"] += 1
        elif char.isdigit():
            result["digits"] += 1
        elif char.isspace():
            result["spaces"] += 1
    
    return result

character_type_analyzer('I am a python developer')

{'vowels': 8, 'consonants': 11, 'digits': 0, 'spaces': 4}

In [3]:
import string

def preprocess(text: str) -> str:
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation
    cleaned_text = "".join(char for char in text if char not in string.punctuation)
    
    return cleaned_text

preprocess('Hi!! How are you doing today?')


'hi how are you doing today'

In [35]:
import pandas as pd

class AttendanceAnalyzer:

    # 1. Create Attendance DataFrame
    def create_attendance_df(self, data: list) -> pd.DataFrame:
        df = pd.DataFrame(data)

    # Fix column name mismatch
        if "Status" in df.columns:
            df = df.rename(columns={"Status": "Attendance"})

    # If Department missing, add default
        if "Department" not in df.columns:
            df["Department"] = "General"

    # Normalize attendance text
        df["Attendance"] = df["Attendance"].str.capitalize()

        return df[["EmployeeID", "Department", "Date", "Attendance"]]

    # 2. Calculate Monthly Attendance Rate per Employee
    def compute_monthly_attendance_rate(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["Month"] = df["Date"].str[:7]  # Extract Year-Month

        total_days = (
            df.groupby(["EmployeeID", "Month"])
              .size()
              .reset_index(name="TotalDays")
        )

        present_df = df[df["Attendance"] == "Present"]

        present_days = (
            present_df.groupby(["EmployeeID", "Month"])
                      .size()
                      .reset_index(name="PresentDays")
        )

        merged = pd.merge(total_days, present_days,
                          on=["EmployeeID", "Month"], how="left")

        merged["PresentDays"] = merged["PresentDays"].fillna(0)
        merged["Attendance Rate"] = (merged["PresentDays"] / merged["TotalDays"]) * 100

        return merged[["EmployeeID", "Month", "Attendance Rate"]]

    # 3. Add Absence Flag Column
    def add_absence_flag(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["IsAbsent"] = (df["Attendance"] == "Absent").astype(int)
        return df

    # 4. Filter Employees with High Absenteeism
    def high_absentees(self, df: pd.DataFrame, threshold: int) -> pd.DataFrame:
        absent_df = df[df["Attendance"] == "Absent"]

        absence_counts = (
            absent_df.groupby("EmployeeID")
                     .size()
                     .reset_index(name="Absence Count")
        )

        return absence_counts[absence_counts["Absence Count"] > threshold]

    # 5. Department-wise Attendance Summary
    def department_attendance_summary(self, df: pd.DataFrame) -> pd.DataFrame:
        ct = pd.crosstab(df["Department"], df["Attendance"])
        ct = ct.reindex(columns=["Present", "Absent", "Leave"], fill_value=0)
        return ct.reset_index()


In [36]:
obj = AttendanceAnalyzer()


In [37]:
attendance_data = [
    {"EmployeeID": "E001", "Name": "Amit Sharma", "Department": "HR",        "Date": "2025-01-01", "Status": "present"},
    {"EmployeeID": "E002", "Name": "Priya Singh", "Department": "Finance",   "Date": "2025-01-01", "Status": "absent"},
    {"EmployeeID": "E003", "Name": "Rahul Verma", "Department": "IT",        "Date": "2025-01-01", "Status": "leave"},
    {"EmployeeID": "E004", "Name": "Kiran Patel",  "Department": "HR",        "Date": "2025-01-01", "Status": "present"},
    {"EmployeeID": "E005", "Name": "Sneha Gupta",  "Department": "Marketing", "Date": "2025-01-01", "Status": "present"},

    {"EmployeeID": "E001", "Name": "Amit Sharma", "Department": "HR",        "Date": "2025-01-02", "Status": "absent"},
    {"EmployeeID": "E002", "Name": "Priya Singh", "Department": "Finance",   "Date": "2025-01-02", "Status": "present"},
    {"EmployeeID": "E003", "Name": "Rahul Verma", "Department": "IT",        "Date": "2025-01-02", "Status": "present"},
    {"EmployeeID": "E006", "Name": "Vikram Rao",  "Department": "IT",        "Date": "2025-01-02", "Status": "leave"},
    {"EmployeeID": "E007", "Name": "Neha Joshi",  "Department": "Marketing", "Date": "2025-01-02", "Status": "present"},

    {"EmployeeID": "E008", "Name": "Sandeep Kumar","Department": "Sales",     "Date": "2025-01-03", "Status": "absent"},
    {"EmployeeID": "E009", "Name": "Rohit Mehta", "Department": "Sales",     "Date": "2025-01-03", "Status": "present"},
    {"EmployeeID": "E010", "Name": "Divya Nair",  "Department": "Finance",   "Date": "2025-01-03", "Status": "leave"},
    {"EmployeeID": "E011", "Name": "Anil Kumar",  "Department": "IT",        "Date": "2025-01-03", "Status": "present"},
    {"EmployeeID": "E012", "Name": "Meera Iyer",  "Department": "HR",        "Date": "2025-01-03", "Status": "present"},

    {"EmployeeID": "E013", "Name": "Rajesh Singh","Department": "Marketing", "Date": "2025-01-04", "Status": "leave"},
    {"EmployeeID": "E014", "Name": "Varun Shah",  "Department": "Finance",   "Date": "2025-01-04", "Status": "absent"},
    {"EmployeeID": "E015", "Name": "Pooja Agarwal","Department": "Sales",     "Date": "2025-01-04", "Status": "present"},
    {"EmployeeID": "E016", "Name": "Harish Reddy","Department": "IT",        "Date": "2025-01-04", "Status": "present"},
    {"EmployeeID": "E017", "Name": "Lakshmi Menon","Department": "HR",       "Date": "2025-01-04", "Status": "present"},

    {"EmployeeID": "E001", "Name": "Amit Sharma", "Department": "HR",        "Date": "2025-01-05", "Status": "present"},
    {"EmployeeID": "E002", "Name": "Priya Singh", "Department": "Finance",   "Date": "2025-01-05", "Status": "present"},
    {"EmployeeID": "E003", "Name": "Rahul Verma", "Department": "IT",        "Date": "2025-01-05", "Status": "absent"},
    {"EmployeeID": "E004", "Name": "Kiran Patel",  "Department": "HR",        "Date": "2025-01-05", "Status": "leave"},
    {"EmployeeID": "E005", "Name": "Sneha Gupta",  "Department": "Marketing", "Date": "2025-01-05", "Status": "present"},

    {"EmployeeID": "E006", "Name": "Vikram Rao",  "Department": "IT",        "Date": "2025-01-06", "Status": "present"},
    {"EmployeeID": "E007", "Name": "Neha Joshi",  "Department": "Marketing", "Date": "2025-01-06", "Status": "absent"},
    {"EmployeeID": "E008", "Name": "Sandeep Kumar","Department": "Sales",     "Date": "2025-01-06", "Status": "present"},
    {"EmployeeID": "E009", "Name": "Rohit Mehta", "Department": "Sales",     "Date": "2025-01-06", "Status": "present"},
    {"EmployeeID": "E010", "Name": "Divya Nair",  "Department": "Finance",   "Date": "2025-01-06", "Status": "absent"}
]


In [38]:
df = obj.create_attendance_df(attendance_data)
df.head()

,EmployeeID,Department,Date,Attendance
0,E001,HR,2025-01-01,Present
1,E002,Finance,2025-01-01,Absent
2,E003,IT,2025-01-01,Leave
3,E004,HR,2025-01-01,Present
4,E005,Marketing,2025-01-01,Present


In [39]:
obj.compute_monthly_attendance_rate(df)

,EmployeeID,Month,Attendance Rate
0,E001,2025-01,66.666667
1,E002,2025-01,66.666667
2,E003,2025-01,33.333333
3,E004,2025-01,50.000000
4,E005,2025-01,100.000000
5,E006,2025-01,50.000000
6,E007,2025-01,50.000000
7,E008,2025-01,50.000000
8,E009,2025-01,100.000000
9,E010,2025-01,0.000000


In [40]:
obj.add_absence_flag(df)

,EmployeeID,Department,Date,Attendance,IsAbsent
0,E001,HR,2025-01-01,Present,0
1,E002,Finance,2025-01-01,Absent,1
2,E003,IT,2025-01-01,Leave,0
3,E004,HR,2025-01-01,Present,0
4,E005,Marketing,2025-01-01,Present,0
5,E001,HR,2025-01-02,Absent,1
6,E002,Finance,2025-01-02,Present,0
7,E003,IT,2025-01-02,Present,0
8,E006,IT,2025-01-02,Leave,0
9,E007,Marketing,2025-01-02,Present,0


In [41]:
obj.high_absentees(df, threshold=2)

,EmployeeID,Absence Count


In [42]:
obj.department_attendance_summary(df)

Attendance,Department,Present,Absent,Leave
0,Finance,2,3,1
1,HR,5,1,1
2,IT,4,1,2
3,Marketing,3,1,1
4,Sales,4,1,0
